# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook guides you in loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

## Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) available at:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure the required library is installed
!pip install mlcroissant pandas --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL for the Croissant schema
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata and print basic description
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

### Additional Metadata
- **Citation**: Mugotitsa, B., Maina, D., Owoko, H., Akinyi, L., Greenfield, J., Todd, J., Kavu, T. and Kiragga, A. 2026. A FAIR2 Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa. Frontiers.
- **License**: https://opendatacommons.org/licenses/by/1-0/
- **Published**: 2026-07-17
- **Keywords**: demographic data, GAD-7, Kenya, Kilifi County, mental health, mental health symptoms, PHQ-9, PSQ


## 2. Data Overview

Review discovered record sets (tables), fields, and entity `@id`s.

**All references to dataset elements use their unique `@id` fields as required by the Croissant standard.**

In [ ]:
# List all record sets and their @id fields
print("Available record sets in the dataset and their @id values:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs['@id'] if '@id' in rs else getattr(rs, '@id', '')}")
    record_set_ids.append(rs['@id'] if '@id' in rs else getattr(rs, '@id', ''))
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print(f"  Fields (@id):")
    for fld in rs.fields:
        print(f"    - {fld['@id'] if '@id' in fld else getattr(fld, '@id', '')}: {fld.name if hasattr(fld, 'name') else ''}")
    print()

**Let's view an example record from each record set.**

In [ ]:
# Display a sample record from each record set
for rs_id in record_set_ids:
    print(f"Sample from record set '@id': {rs_id}")
    try:
        sample = next(dataset.records(record_set=rs_id))
        print({k: v for k, v in sample.items()})
    except Exception as e:
        print(f"  Could not retrieve sample: {str(e)}")
    print('-' * 60)

## 3. Data Extraction

Let's extract the primary survey data from the record set containing the main survey responses. We continue to reference all entities by their `@id` fields.

In [ ]:
# For this dataset, we use all record sets to extract their data into DataFrames
# As an example, we focus on the main record set for the analysis (usually 'survey' or similar; adapt as needed)
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '@id': {rs_id}; columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {str(e)}\n")

Let's pick one main DataFrame for deeper inspection. For demonstration, we will select the first available record set.

In [ ]:
# Pick a record set for deeper analysis
main_rs_id = record_set_ids[0]  # adjust index if required
df_main = dataframes[main_rs_id]
print(f"First 5 records from record set '@id': {main_rs_id}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing such as filtering, normalizing, and grouping. 

We'll select a numeric field for demonstration (e.g., the GAD-7 score field, commonly named `gad_7_total` or similar; adjust `numeric_field` as needed based on the column list).

In [ ]:
# Choose a numeric field for the demonstration
possible_numeric_cols = [col for col in df_main.columns if df_main[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df_main[col])]
print(f"Possible numeric fields in main record set: {possible_numeric_cols}")

# Try typical clinical scales names
for candidate in ['gad_7_total', 'phq_9_total', 'psq_total', 'GAD-7', 'PHQ-9']:
    if candidate in df_main.columns:
        numeric_field = candidate
        break
else:
    numeric_field = possible_numeric_cols[0] if possible_numeric_cols else None
    if not numeric_field:
        raise ValueError("No numeric field found in the main record set.")
print(f"Numeric field selected for demo: {numeric_field}")

# Filter records with score > threshold
threshold = 10
filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records from '{main_rs_id}' with {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records (first 5):")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by an attribute, e.g., by 'gender' if present
group_field = None
for candidate in ['gender', 'sex', 'Gender']:
    if candidate in df_main.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
    print(f"\nGrouped mean {numeric_field} by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its values by group (e.g., by gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df_main[numeric_field].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if 'group_field' exists
if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df_main)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- We demonstrated how to load and explore the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library, referencing all record sets and fields by their stable Croissant `@id` identifiers.
- The main record set includes demographic and mental health assessment data enabling statistical and visual analysis.
- For deeper use (cleaning, modeling), consult the record set and field documentation in the [Croissant schema source](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

**Next steps:** You may further apply advanced analysis, machine learning models, or join other record sets as documented, always referencing objects by `@id` for reproducibility and schema-alignment.